In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType
from pyspark.sql.window import Window
 
print("=" * 60)
print("SILVER CLEANING — Healthcare Patient Encounters")
print("=" * 60)

StatementMeta(, 9aeeb90f-7e68-438b-8cb3-536e486f3f19, 3, Finished, Available, Finished, False)

SILVER CLEANING — Healthcare Patient Encounters


## Load Bronze

In [2]:
df = spark.read.format("delta").table("bronze_encounters")
total_raw = df.count()
print(f"Raw encounters : {total_raw:,}")
print(f"Columns        : {len(df.columns)}")

StatementMeta(, 9aeeb90f-7e68-438b-8cb3-536e486f3f19, 4, Finished, Available, Finished, False)

Raw encounters : 101,766
Columns        : 50


## Replace '?' with null

In [3]:
Q_COLS = ["race","diag_1","diag_2","diag_3",
          "payer_code","medical_specialty","weight"]
for col in Q_COLS:
    if col in df.columns:
        df = df.withColumn(
            col, F.when(F.col(col) == "?", None).otherwise(F.col(col))
        )
q_count = df.filter(F.col("race").isNull()).count()
print(f"\n'?' replaced → nulls now visible. Example: race nulls = {q_count:,}")

StatementMeta(, 9aeeb90f-7e68-438b-8cb3-536e486f3f19, 5, Finished, Available, Finished, False)


'?' replaced → nulls now visible. Example: race nulls = 2,273


In [4]:
display(df.limit(20))

StatementMeta(, 9aeeb90f-7e68-438b-8cb3-536e486f3f19, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4c5ad908-22e3-4269-85d4-6815ae478d55)

##  Drop high-null columns

In [5]:
# weight: 97% null — not usable for ML
# payer_code: 40% null (keep for potential insurance analysis)
# medical_specialty: 49% null (keep — useful for ward RLS filter)
df = df.drop("weight")
print("Dropped: weight (97% null)")

StatementMeta(, 9aeeb90f-7e68-438b-8cb3-536e486f3f19, 7, Finished, Available, Finished, False)

Dropped: weight (97% null)


## Cast numeric columns

In [6]:
INT_COLS = ["time_in_hospital","num_lab_procedures","num_procedures",
            "num_medications","number_diagnoses","number_inpatient",
            "number_emergency","number_outpatient","encounter_id","patient_nbr"]
for col in INT_COLS:
    if col in df.columns:
        df = df.withColumn(col, F.col(col).cast(IntegerType()))
print(f"Cast {len(INT_COLS)} numeric columns to IntegerType")

StatementMeta(, 9aeeb90f-7e68-438b-8cb3-536e486f3f19, 8, Finished, Available, Finished, False)

Cast 10 numeric columns to IntegerType


## Deduplicate — one row per patient

In [7]:
# Some patients have multiple encounters in the dataset.
# For readmission prediction: keep the most recent encounter
# (higher encounter_id = more recent in this dataset).
w_pat = Window.partitionBy("patient_nbr").orderBy(F.col("encounter_id").desc())
df = (df
    .withColumn("enc_rank", F.rank().over(w_pat))
    .filter(F.col("enc_rank") == 1)
    .drop("enc_rank"))
 
n_dedup = df.count()
print(f"\nDuplicate encounters removed : {total_raw - n_dedup:,}")
print(f"Unique patients remaining    : {n_dedup:,}")

StatementMeta(, 9aeeb90f-7e68-438b-8cb3-536e486f3f19, 9, Finished, Available, Finished, False)


Duplicate encounters removed : 30,248
Unique patients remaining    : 71,518


## Encode age band → ordinal integer

In [8]:
# ML models need numeric inputs. Age brackets encode
# naturally as ordinals 0–9 without losing the order.
AGE_MAP = {"[0-10)":0,"[10-20)":1,"[20-30)":2,"[30-40)":3,
           "[40-50)":4,"[50-60)":5,"[60-70)":6,
           "[70-80)":7,"[80-90)":8,"[90-100)":9}
age_expr = F.col("age")
for band, val in AGE_MAP.items():
    age_expr = F.when(F.col("age") == band, val).otherwise(age_expr)
df = df.withColumn("age_ord", age_expr.cast(IntegerType()))
print("\nAge bands encoded to ordinal integers (0–9)")

StatementMeta(, 9aeeb90f-7e68-438b-8cb3-536e486f3f19, 10, Finished, Available, Finished, False)


Age bands encoded to ordinal integers (0–9)


## Create binary classification target

In [9]:
# Original column: '<30' (readmit within 30 days), '>30', 'NO'
# We create a clean binary label for XGBoost.
df = df.withColumn(
    "readmitted_30d",
    F.when(F.col("readmitted") == "<30", 1).otherwise(0)
)
pos_rate = df.agg(
    F.avg(F.col("readmitted_30d").cast(FloatType()))
).collect()[0][0]
 
print(f"\nTarget variable distribution:")
df.groupBy("readmitted").count().orderBy("count", ascending=False).show()
print(f"30-day readmission rate : {pos_rate:.1%}")
print(f"Class imbalance ratio   : 1:{int(1/pos_rate)} (positive:negative)")
print("→ Use scale_pos_weight in XGBoost to handle this")

StatementMeta(, 4d26aaf9-db8f-4052-bf56-49fe1f397735, 11, Finished, Available, Finished, False)


Target variable distribution:
+----------+-----+
|readmitted|count|
+----------+-----+
|        NO|54374|
|       >30|13920|
|       <30| 3224|
+----------+-----+

30-day readmission rate : 4.5%
Class imbalance ratio   : 1:22 (positive:negative)
→ Use scale_pos_weight in XGBoost to handle this


## Null audit

In [10]:
null_counts = [
    (c, df.filter(F.col(c).isNull()).count())
    for c in df.columns
]
null_counts.sort(key=lambda x: -x[1])
for col_name, cnt in null_counts[:10]:
    pct = cnt / n_dedup * 100
    flag = "⚠️ " if pct > 5 else "✅"
    print(f"  {flag} {col_name:<35} {cnt:>6,} ({pct:.1f}%)")

StatementMeta(, 4d26aaf9-db8f-4052-bf56-49fe1f397735, 12, Finished, Available, Finished, False)

  ⚠️  medical_specialty                   34,525 (48.3%)
  ⚠️  payer_code                          30,085 (42.1%)
  ✅ race                                 1,878 (2.6%)
  ✅ diag_3                               1,146 (1.6%)
  ✅ diag_2                                 290 (0.4%)
  ✅ diag_1                                  17 (0.0%)
  ✅ encounter_id                             0 (0.0%)
  ✅ patient_nbr                              0 (0.0%)
  ✅ gender                                   0 (0.0%)
  ✅ age                                      0 (0.0%)


## Write Silver table

In [11]:
df.write.format("delta").mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable("silver_encounters_clean")
 
print(f"\n silver_encounters_clean: {df.count():,} rows, {len(df.columns)} cols")
print("Proceed to → 03_feature_engineering.py")

StatementMeta(, 4d26aaf9-db8f-4052-bf56-49fe1f397735, 13, Finished, Available, Finished, False)


 silver_encounters_clean: 71,518 rows, 51 cols
Proceed to → 03_feature_engineering.py


In [ ]:
# Notebook: 02_silver_cleaning.py
# Scenario 02: Healthcare Patient Readmission Prediction
# Input:  bronze_encounters (101,766 rows)
# Output: silver_encounters_clean
# Attach to: PatientReadmissionPrediction